# Pipeline data viewer
Browse the saved output of every stage, from collection to LLM verification.

In [3]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "data"

# Set to a run id (folder in data/runs/) to view an older run, or None for the latest output.
RUN_ID = None


def load(stage, name):
    base = DATA / "runs" / RUN_ID if RUN_ID else DATA
    return json.loads((base / stage / f"{name}.json").read_text())


def frame(rows, cols=None):
    df = pd.DataFrame(rows)
    return df[[c for c in cols if c in df.columns]] if cols else df


def show(d):
    """Pretty-print one record."""
    print(json.dumps(d, indent=2, ensure_ascii=False, default=str))

print("Data folder:", DATA)
print("Runs:", sorted(p.name for p in (DATA / "runs").glob("*"))[-5:])

Data folder: /Users/apple/Documents/ca-mva-fatality-reporting/data
Runs: ['20261014T010543Z']


## 1. source_collection
### 1a. `raw_feed_items.json` — every feed item before any filter

In [4]:
raw = load("source_collection", "raw_feed_items")
print(raw["run_at"], "| total items:", raw["total_items"])
raw_df = frame(raw["items"])
display(raw_df[["title", "published", "feed_name", "query"]].head(50))
display(raw_df["feed_name"].value_counts().rename("items per feed").to_frame())

2026-10-14T01:05:43.868457+00:00 | total items: 433


,title,published,feed_name,query
0,"Langer's Deli owner blasts $1,800 trash fee tied to dumpster lock","Wed, 14 Oct 2026 00:31:05 +0000",ABC7 Los Angeles,None
1,Legendary singer-songwriter Joni Mitchell becomes US citizen,"Wed, 14 Oct 2026 00:07:44 +0000",ABC7 Los Angeles,None
2,Michael Jackson's 1993 Super Bowl performance is subject of new Hulu documen...,"Tue, 13 Oct 2026 23:22:28 +0000",ABC7 Los Angeles,None
3,Aerial view shows extent of destruction to homes along Long Beach coast,"Tue, 13 Oct 2026 23:12:58 +0000",ABC7 Los Angeles,None
4,"Defendant's palm print found on bottle at Faith Hedgepeth crime scene, exper...","Tue, 13 Oct 2026 23:11:37 +0000",ABC7 Los Angeles,None
5,Police identify 42-year-old man killed after being ejected from Six Flags Ov...,"Tue, 13 Oct 2026 22:49:04 +0000",ABC7 Los Angeles,None
6,"Double homicide being investigated at Great Wolf Lodge in Scotrun, Pennsylva...","Tue, 13 Oct 2026 22:33:29 +0000",ABC7 Los Angeles,None
7,Bay Area woman Rong Yuan poisoned by co-worker David Xu speaks out for 1st t...,"Tue, 13 Oct 2026 21:54:33 +0000",ABC7 Los Angeles,None
8,Pope surgery update: Vatican says Pope Leo XIV has lung cancer and the lower...,"Tue, 13 Oct 2026 21:40:20 +0000",ABC7 Los Angeles,None
9,Anthropic warns AI models could soon copy themselves in small devices,"Tue, 13 Oct 2026 21:17:24 +0000",ABC7 Los Angeles,None


,items per feed
feed_name,
"Google News: ""California"" pedestrian killed crash",65
"Google News: ""California Highway Patrol"" fatal collision",59
"Google News: ""California"" killed in crash",53
"Google News: ""California"" motorcyclist killed crash",53
KRON4,50
"Google News: ""California"" fatal crash",45
FOX 5 San Diego,21
ABC7 Los Angeles,20
KCRA Sacramento,20


### 1b. `fetched_sources.json` — per-source results and every record with its filter decision

In [5]:
fetched = load("source_collection", "fetched_sources")
print("lookback hours:", fetched["lookback_hours"])
show(fetched["summary"])
display(frame(fetched["sources"], ["source_name", "source_type", "http_status", "error", "items_seen", "candidate_count", "records_kept", "result"]))

lookback hours: 168
{
  "sources_requested": 15,
  "sources_failed": 0,
  "items_seen": 533,
  "items_skipped_by_filters": 230,
  "duplicates_merged": 79,
  "collected_records": 136,
  "records_by_decision": {
    "consider": 8,
    "review": 127,
    "discard": 1
  },
  "discarded_by_reason": {
    "story says the crash happened earlier": 1
  }
}


,source_name,source_type,http_status,error,items_seen,candidate_count,records_kept,result
0,San Bernardino County Coroner,coroner,200,None,54,0,0,no_relevant_items
1,Riverside County Sheriff-Coroner,coroner,200,None,36,2,2,contributed
2,California Highway Patrol,official_listing,200,None,10,0,0,no_relevant_items
3,ABC7 Los Angeles,rss,200,None,20,0,0,no_relevant_items
4,KTLA,rss,200,None,19,0,0,no_relevant_items
5,KCRA Sacramento,rss,200,None,20,0,0,no_relevant_items
6,FOX 5 San Diego,rss,200,None,21,0,0,no_relevant_items
7,KRON4,rss,200,None,50,1,1,contributed
8,Victor Valley News,rss,200,None,10,3,3,contributed
9,"Google News: ""California"" fatal crash",google_news_rss,200,None,45,31,30,contributed


In [6]:
records_df = frame(fetched["collected_records"])
display(records_df[["title", "publisher", "source_type", "incident_date", "county", "place", "status", "selection_bucket", "selection_reason"]])
display(records_df["status"].value_counts().rename("records by status").to_frame())

,title,publisher,source_type,incident_date,county,place,status,selection_bucket,selection_reason
0,Riverside County Coroner file 2026-07531,Riverside County Sheriff-Coroner,coroner,2026-10-10T07:00:00+00:00,Riverside,Riverside,recent,consider,official coroner record of a road death in the window
1,Riverside County Coroner file 2026-07406,Riverside County Sheriff-Coroner,coroner,2026-10-06T07:00:00+00:00,Riverside,Moreno Valley,recent,consider,official coroner record of a road death in the window
2,Motorcyclist killed in crash on I-80 near MacArthur Maze,KRON4,rss,2026-10-13T07:00:00+00:00,None,None,recent,consider,crash date in story text is inside the window
3,Apple Valley Man Identified as Driver Killed in Crash Involving Semi-Truck o...,Victor Valley News,rss,2026-10-09T07:00:00+00:00,San Bernardino,Apple Valley,recent,consider,crash date in story text is inside the window
4,Husband And Wife From Hesperia Killed After Vehicle Flew Off Cliff In The Ca...,Victor Valley News,rss,2026-10-11T07:00:00+00:00,San Bernardino,Hesperia,recent,consider,crash date in story text is inside the window
...,...,...,...,...,...,...,...,...,...
131,Motorcyclist Dies in Crash Near Emeryville's MacArthur Maze,Hoodline,google_news_rss,None,None,None,incident_date_unverified,review,"crash date not found in story text, California location not confirmed"
132,Coroner: Cause of death for driver killed in fiery vehicle crash at hotel pa...,KBAK,google_news_rss,None,None,None,incident_date_unverified,review,"crash date not found in story text, California location not confirmed"
133,"Riverside Woman, 20, Dies After Being Ejected From Jeep in Dry Lakebed Crash...",Instagram,google_news_rss,None,San Bernardino,San Bernardino,incident_date_unverified,review,crash date not found in story text
134,"Martiniano Zamora killed in Taft two-vehicle crash, coroner says",The Legal Advocate,google_news_rss,None,None,None,incident_date_unverified,review,"crash date not found in story text, California location not confirmed"


,records by status
status,
incident_date_unverified,127
recent,8
review_location,1


### 1c. `sources_to_open.json` — one entry per crash (confirmed + doubtful)

In [7]:
to_open = load("source_collection", "sources_to_open")
show(to_open["summary"])
cols = ["case_id", "confidence", "title", "publisher", "incident_date", "location", "victim_type", "victim_age", "why"]
print("Confirmed")
display(frame(to_open["confirmed"], cols))
print("Doubtful")
display(frame(to_open["doubtful"], cols))

{
  "confirmed_cases": 7,
  "doubtful_cases": 101,
  "urls_in_confirmed": 12,
  "urls_in_doubtful": 125,
  "doubtful_dropped_by_reason": {
    "follow-up story (court/sentencing/lawsuit) about an earlier crash": 1
  }
}
Confirmed


,case_id,confidence,title,publisher,incident_date,location,victim_type,victim_age,why
0,case-b407d1a677,confirmed,Riverside County Coroner file 2026-07531,Riverside County Sheriff-Coroner,2026-10-10T07:00:00+00:00,"{'county': 'Riverside', 'place': 'Riverside', 'road': None, 'location_text':...",None,26.0,official coroner record of a road death in the window
1,case-b1707d019a,confirmed,Riverside County Coroner file 2026-07406,Riverside County Sheriff-Coroner,2026-10-06T07:00:00+00:00,"{'county': 'Riverside', 'place': 'Moreno Valley', 'road': None, 'location_te...",None,68.0,official coroner record of a road death in the window
2,case-29aa985ce7,confirmed,Motorcyclist killed in crash on I-80 near MacArthur Maze,KRON4,2026-10-13T07:00:00+00:00,"{'county': None, 'place': None, 'road': 'I-80', 'location_text': None, 'loca...",motorcyclist,NaN,crash date in story text is inside the window
3,case-cea0ac6345,confirmed,Apple Valley Man Identified as Driver Killed in Crash Involving Semi-Truck o...,Victor Valley News,2026-10-09T07:00:00+00:00,"{'county': 'San Bernardino', 'place': 'Apple Valley', 'road': None, 'locatio...",driver,32.0,crash date in story text is inside the window
4,case-179eda74a4,confirmed,Husband And Wife From Hesperia Killed After Vehicle Flew Off Cliff In The Ca...,Victor Valley News,2026-10-11T07:00:00+00:00,"{'county': 'San Bernardino', 'place': 'Hesperia', 'road': '15 Freeway', 'loc...",passenger,35.0,crash date in story text is inside the window
5,case-6459ec32b9,confirmed,Semi driver from California dies in Wednesday crash on I-35 in Olathe,WIBW,2026-10-07T07:00:00+00:00,"{'county': None, 'place': None, 'road': 'I-35', 'location_text': None, 'loca...",driver,NaN,crash date in story text is inside the window
6,case-4d4b12fbbe,confirmed,"Riverside, CA – Motorcyclist Dylan Monterola-Alas Dies in Crash on Martin Lu...",Facebook,2026-10-10T07:00:00+00:00,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",motorcyclist,NaN,crash date in story text is inside the window


Doubtful


,case_id,confidence,title,publisher,incident_date,location,victim_type,victim_age,why
0,case-2d770cf204,doubtful,Pedestrian killed on Lavista Road Tuesday morning,FOX 5 Atlanta,2026-10-13T07:00:00+00:00,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",pedestrian,NaN,California location not confirmed
1,case-74ae5f6020,doubtful,"Young Southern California woman, 19, killed, 2 injured in single-car crash i...",KTLA,None,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",None,19.0,crash date not found in story text
2,case-d1896dcfaa,doubtful,"Fatal crash shuts down SB U.S. 101 near Gilroy, CHP says",NBC Bay Area,None,"{'county': None, 'place': None, 'road': 'U.S. 101', 'location_text': None, '...",None,NaN,crash date not found in story text
3,case-ace36d19da,doubtful,Fatal crash on U.S. 395 near Constantia Road in California,KTVN,None,"{'county': None, 'place': None, 'road': 'U.S. 395', 'location_text': None, '...",None,NaN,crash date not found in story text
4,case-1a7322ae1f,doubtful,One killed after sheriff's deputy collision sets off chain-reaction crash in...,Los Angeles Times,None,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",None,NaN,"crash date not found in story text, California location not confirmed"
...,...,...,...,...,...,...,...,...,...
96,case-ea511c3962,doubtful,"Motorcyclist Killed in Solo Crash West of Grass Valley, CHP Confirms",Hoodline,None,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",motorcyclist,NaN,crash date not found in story text
97,case-ed52f33dcf,doubtful,Motorcyclist Dies in Crash Near Emeryville's MacArthur Maze,Hoodline,None,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",motorcyclist,NaN,"crash date not found in story text, California location not confirmed"
98,case-1461bcfb5e,doubtful,Coroner: Cause of death for driver killed in fiery vehicle crash at hotel pa...,KBAK,None,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",driver,NaN,"crash date not found in story text, California location not confirmed"
99,case-11879e9617,doubtful,"Martiniano Zamora killed in Taft two-vehicle crash, coroner says",The Legal Advocate,None,"{'county': None, 'place': None, 'road': None, 'location_text': None, 'locati...",None,NaN,"crash date not found in story text, California location not confirmed"


## 2. link_extraction
### `articles.json` — fetched article text, page metadata, full-text filter decision

In [8]:
art = load("link_extraction", "articles")
show(art["summary"])
articles_df = frame(art["articles"])
articles_df["text_chars"] = articles_df["text"].fillna("").str.len()
display(articles_df[["article_id", "case_id", "title", "publisher", "fetch_status", "extraction_method",
                     "text_chars", "incident_date", "county", "filter_decision", "filter_reasons"]])
display(articles_df.groupby(["fetch_status", "filter_decision"], dropna=False).size().rename("articles").to_frame())

{
  "cases": 108,
  "articles": 135,
  "unique_urls_fetched": 135,
  "google_news_links_decoded": "131/131",
  "by_fetch_status": {
    "ok": 108,
    "blocked": 20,
    "failed": 6,
    "too_short": 1
  },
  "by_filter_decision": {
    "keep": 114,
    "drop": 21
  },
  "drop_reasons": {
    "crash date in the article is outside the lookback window": 7,
    "article says the crash happened earlier": 12,
    "article places the crash outside California (Michigan)": 1,
    "article places the crash outside California (North Dakota)": 1,
    "article places the crash outside California (Idaho)": 1
  },
  "cases_without_text": 13
}


,article_id,case_id,title,publisher,fetch_status,extraction_method,text_chars,incident_date,county,filter_decision,filter_reasons
0,art-a16522296df1,case-b407d1a677,Riverside County Coroner file 2026-07531,Riverside County Sheriff-Coroner,ok,page,690,2026-10-10T07:00:00+00:00,Riverside,keep,[]
1,art-5a75f6cf9c72,case-b1707d019a,Riverside County Coroner file 2026-07406,Riverside County Sheriff-Coroner,ok,page,700,2026-10-06T07:00:00+00:00,Riverside,keep,[]
2,art-29aa985ce7c9,case-29aa985ce7,Motorcyclist killed in crash on I-80 near MacArthur Maze,KRON4,blocked,None,0,2026-10-13T07:00:00+00:00,None,keep,[]
3,art-abbdaebbe561,case-29aa985ce7,Motorcyclist killed in crash on westbound I-80 in Emeryville,NBC Bay Area,ok,page,508,2026-10-13T07:00:00+00:00,None,keep,[]
4,art-cea0ac6345f9,case-cea0ac6345,Apple Valley Man Identified as Driver Killed in Crash Involving Semi-Truck o...,Victor Valley News,ok,page,1562,2026-10-09T07:00:00+00:00,San Bernardino,keep,[]
...,...,...,...,...,...,...,...,...,...,...,...
130,art-ff85d62a4561,case-ea511c3962,"Motorcyclist Killed in Solo Crash West of Grass Valley, CHP Confirms",Hoodline,ok,page,2205,None,Nevada,drop,[article says the crash happened earlier]
131,art-3eddd4f642ca,case-ed52f33dcf,Motorcyclist Dies in Crash Near Emeryville's MacArthur Maze,Hoodline,ok,page,3214,None,None,drop,[article says the crash happened earlier]
132,art-15797366b714,case-1461bcfb5e,Coroner: Cause of death for driver killed in fiery vehicle crash at hotel pa...,KBAK,ok,page,1029,None,Kern,keep,"[crash date not found in story text, conflicting crash dates in story text: ..."
133,art-98028faad86d,case-11879e9617,"Martiniano Zamora killed in Taft two-vehicle crash, coroner says",The Legal Advocate,ok,page,2866,2026-09-26T07:00:00+00:00,Kern,drop,[crash date in the article is outside the lookback window]


articles
fetch_status filter_decision          
blocked      keep                   20
failed       keep                    6
ok           drop                   21
             keep                   87
too_short    keep                    1

In [9]:
# Read one article: change the id
ARTICLE_ID = articles_df["article_id"].iloc[0]
a = articles_df[articles_df["article_id"] == ARTICLE_ID].iloc[0].to_dict()
text = a.pop("text") or ""
show(a)
print("\n--- TEXT ---\n", text[:3000])

{
  "article_id": "art-a16522296df1",
  "case_id": "case-b407d1a677",
  "case_confidence": "confirmed",
  "case_incident_date": "2026-10-10T07:00:00+00:00",
  "url": "https://www.riversidesheriff.org/m/newsflash/Home/Detail/7410",
  "rss_url": "https://www.riversidesheriff.org/m/newsflash/Home/Detail/7410",
  "title": "Riverside County Coroner file 2026-07531",
  "description": "Coroner's File #: 2026-07531 | Last Name: Monterola-Alas | Gender: Male | First Name: Dylan | Age: 26 Years | Next of Kin Notified: Yes | Date of Death: 10/10/2026 | Time of Death: 1942 Hours | Date of Injury: 10/10/2026 | Time of Injury: 1902 Hours | Location of Death: Riverside Community Hospital, Magnolia Ave., Riverside, CA 92501 | Location of Injury: Roadway, Martin Luther King Blvd., approx. 53 ft. E/O Angelo St., Riverside, CA 92507 | Agency Investigating: Riverside Police Department | Investigation #: 260027175 | Comments: Refer to investigating agency for further information. | Initial, Update or Final

### `extraction_errors.json`

In [10]:
errors = load("link_extraction", "extraction_errors")
display(frame(errors, ["case_id", "article_id", "title", "fetch_status", "error"]))

,case_id,article_id,title,fetch_status,error
0,case-29aa985ce7,art-29aa985ce7c9,Motorcyclist killed in crash on I-80 near MacArthur Maze,blocked,HTTP 403: Forbidden
1,case-4d4b12fbbe,art-4d4b12fbbe4a,"Riverside, CA – Motorcyclist Dylan Monterola-Alas Dies in Crash on Martin Lu...",failed,HTTP 400: Bad Request
2,case-74ae5f6020,art-74ae5f6020ee,"Young Southern California woman, 19, killed, 2 injured in single-car crash i...",blocked,HTTP 403: Forbidden
3,case-18e29bb5b5,art-18e29bb5b5f3,"1 killed, 1 injured in 2-vehicle crash in Davis",failed,HTTP 406: Not Acceptable
4,case-fcf7365bd9,art-fcf7365bd968,Fatal crash shuts down all southbound I-805 lanes in South Bay,blocked,HTTP 403: Forbidden
5,case-bb8eb6cbe2,art-bb8eb6cbe238,Fatal crash under investigation by CHP,blocked,HTTP 403: Forbidden
6,case-7e4299b3fd,art-7e4299b3fd4b,Motorcyclist fatally ejected during Mercedes-Benz crash: CHP Santa Rosa,blocked,HTTP 403: Forbidden
7,case-6013bb7e2b,art-99494dfb1444,84-year-old California woman identified in fatal Honolulu parking garage crash,failed,HTTP 429: Too Many Requests
8,case-8eaf556cde,art-14e8a34252c5,2 killed in head-on crash on Highway 99 in Sutter County,blocked,HTTP 403: Forbidden
9,case-8eaf556cde,art-909ae383a6ca,"Fatal Head-on Crash Highway 99, Howsley Road, Pleasant Grove",blocked,HTTP 403: Forbidden


## 3. incident_grouping
### `incidents.json` — articles grouped per crash

In [11]:
inc = load("incident_grouping", "incidents")
show(inc["summary"])
inc_df = frame(inc["incidents"])
inc_df["n_conflicts"] = inc_df["conflicts"].str.len()
inc_df["n_uncertainties"] = inc_df["uncertainties"].str.len()
display(inc_df[["incident_id", "candidate_incident_dates", "locations", "victim_types_mentioned", "victim_ages_mentioned",
                "article_count", "articles_with_text", "publishers", "n_conflicts", "n_uncertainties"]])

{
  "articles_in": 135,
  "articles_kept": 114,
  "incidents": 76,
  "multi_article_incidents": 23,
  "incidents_without_text": 13
}


,incident_id,candidate_incident_dates,locations,victim_types_mentioned,victim_ages_mentioned,article_count,articles_with_text,publishers,n_conflicts,n_uncertainties
0,inc-64c24e170cb1,"[{'date': '2026-10-13', 'basis': 'relative_to_publication', 'text': 'Tuesday...","[{'county': None, 'place': None, 'road': 'I-80', 'location_text': None, 'loc...","[bicyclist, driver, motorcyclist]",[],4,3,"[East Bay Times, KRON4, NBC Bay Area, SFGATE]",1,0
1,inc-273f6dde6241,"[{'date': '2026-10-13', 'basis': 'relative_to_publication', 'text': 'Tuesday...","[{'county': None, 'place': None, 'road': 'I-85', 'location_text': None, 'loc...","[driver, pedestrian]",[],1,1,[FOX 5 Atlanta],0,1
2,inc-043fce981ac8,"[{'date': '2026-10-13', 'basis': 'relative_to_publication', 'text': 'Tuesday...","[{'county': 'Los Angeles', 'place': 'Pomona', 'road': '60 Freeway', 'locatio...",[],[62],1,1,[MyNewsLA.com],0,0
3,inc-8f6f56ae3e27,"[{'date': '2026-10-12', 'basis': 'relative_to_publication', 'text': 'Monday'...","[{'county': None, 'place': 'Moreno Valley', 'road': None, 'location_text': N...","[bicyclist, driver, motorcyclist]",[26],1,1,[Patch],0,0
4,inc-8dd3b36c31ea,"[{'date': '2026-10-11', 'basis': 'relative_to_publication', 'text': 'Sunday ...","[{'county': 'San Bernardino', 'place': 'Hesperia', 'road': '15 Freeway', 'lo...","[driver, passenger]","[9, 13, 33, 35]",2,2,[Victor Valley News],1,0
...,...,...,...,...,...,...,...,...,...,...
71,inc-e77b44b6a146,[],"[{'county': 'Los Angeles', 'place': 'Los Angeles', 'road': None, 'location_t...","[driver, pedestrian]",[37],1,1,[The Legal Advocate],0,2
72,inc-7d442f702ccb,[],[],"[bicyclist, motorcyclist]",[],1,0,[Merced Sun-Star],0,2
73,inc-0da8b8d90905,[],"[{'county': 'Stanislaus', 'place': None, 'road': None, 'location_text': None...",[motorcyclist],[53],1,0,[FOX40],0,2
74,inc-d6640bdc7cf8,[],[],"[bicyclist, motorcyclist]",[],1,1,[San Francisco Chronicle],0,2


In [12]:
# Inspect one incident: change the index
i = inc["incidents"][0]
show({k: v for k, v in i.items() if k != "articles"})
display(frame(i["articles"]))

{
  "incident_id": "inc-64c24e170cb1",
  "source_case_ids": [
    "case-057626885f",
    "case-29aa985ce7",
    "case-8c970f7cc4"
  ],
  "candidate_incident_dates": [
    {
      "date": "2026-10-13",
      "basis": "relative_to_publication",
      "text": "Tuesday morning",
      "evidence": "Motorcyclist killed in crash on I-80 near MacArthur Maze (BCN) -- A motorcyclist died in a crash on westbound Interstate 80 in Emeryville near the MacArthur Maze on Tuesday morning, according to the California Highway Patrol.",
      "article_id": "art-29aa985ce7c9"
    },
    {
      "date": "2026-10-13",
      "basis": "relative_to_publication",
      "text": "Tuesday morning",
      "evidence": "Motorcyclist killed in crash on westbound I-80 in Emeryville A motorcyclist died in a crash on westbound Interstate Highway 80 in Emeryville near the MacArthur Maze on Tuesday morning, according to the California Highway Patrol.",
      "article_id": "art-abbdaebbe561"
    },
    {
      "date": "2026-

,article_id,case_id,case_confidence,case_incident_date,url,rss_url,title,description,publisher,published_at,source_type,guid,discovered_by_queries,discovered_by_sources,resolved_url,final_url,canonical_url,fetch_status,error,extraction_method,text_available,text,metadata,case_number,location_status,location_evidence,county,place,road,published_at_resolved,incident_date,incident_date_text,incident_date_basis,incident_date_evidence,incident_date_note,incident_date_note_type,status,filter_decision,filter_reasons,filter_basis
0,art-29aa985ce7c9,case-29aa985ce7,confirmed,2026-10-13T07:00:00+00:00,https://www.kron4.com/news/traffic/motorcyclist-killed-in-crash-on-i-80-near...,https://www.kron4.com/news/traffic/motorcyclist-killed-in-crash-on-i-80-near...,Motorcyclist killed in crash on I-80 near MacArthur Maze,(BCN) -- A motorcyclist died in a crash on westbound Interstate 80 in Emeryv...,KRON4,2026-10-13T15:11:05+00:00,rss,https://www.kron4.com/?p=3156708,"[""California"" motorcyclist killed crash]","[KRON4, Google News: ""California"" motorcyclist killed crash]",https://www.kron4.com/news/traffic/motorcyclist-killed-in-crash-on-i-80-near...,None,https://kron4.com/news/traffic/motorcyclist-killed-in-crash-on-i-80-near-mac...,blocked,HTTP 403: Forbidden,None,False,None,{},None,california_confirmed,California,None,None,I-80,2026-10-13T15:11:05+00:00,2026-10-13T07:00:00+00:00,Tuesday morning,relative_to_publication,Motorcyclist killed in crash on I-80 near MacArthur Maze (BCN) -- A motorcyc...,None,None,recent,keep,[],rss_metadata
1,art-abbdaebbe561,case-29aa985ce7,confirmed,2026-10-13T07:00:00+00:00,https://www.nbcbayarea.com/news/local/emeryville-i-80-motorcyclist-killed-cr...,https://news.google.com/rss/articles/CBMikgFBVV95cUxNdGtxZ28zWFBNcXV2b2piZmc...,Motorcyclist killed in crash on westbound I-80 in Emeryville,The crash was reported shortly after 6:45 a.m. on westbound I-80 west of Pow...,NBC Bay Area,2026-10-13T15:20:30+00:00,google_news_rss,CBMikgFBVV95cUxNdGtxZ28zWFBNcXV2b2piZmcyOGoxMUZ5TmVwajB5clNzRl94S3dJWnowUFpI...,"[""California"" killed in crash, ""California Highway Patrol"" fatal collision]","[Google News: ""California"" killed in crash, Google News: ""California Highway...",https://www.nbcbayarea.com/news/local/emeryville-i-80-motorcyclist-killed-cr...,https://www.nbcbayarea.com/news/local/emeryville-i-80-motorcyclist-killed-cr...,https://nbcbayarea.com/news/local/emeryville-i-80-motorcyclist-killed-crash/...,ok,None,page,True,A motorcyclist died in a crash on westbound Interstate Highway 80 in Emeryvi...,{'page_title': 'Motorcyclist killed in crash on westbound I-80 in Emeryville...,None,california_confirmed,California,None,None,I-80,2026-10-13T15:20:30+00:00,2026-10-13T07:00:00+00:00,Tuesday morning,relative_to_publication,Motorcyclist killed in crash on westbound I-80 in Emeryville A motorcyclist ...,None,None,recent,keep,[],article_text
2,art-c8c84942804e,case-057626885f,doubtful,None,https://www.eastbaytimes.com/2026/10/13/motorcyclist-killed-in-east-bay-free...,https://news.google.com/rss/articles/CBMilwFBVV95cUxQNnlaNDBUbnZlSmRQWjh2dTB...,Motorcyclist killed in East Bay freeway collision,A motorcyclist died after a collision on I-80 in Emeryville Tuesday morning.,East Bay Times,2026-10-13T17:01:31+00:00,google_news_rss,CBMilwFBVV95cUxQNnlaNDBUbnZlSmRQWjh2dTByV3puazBvaTBWX2hDZk43c21OLS0wVko4MzVS...,"[""California"" motorcyclist killed crash]","[Google News: ""California"" motorcyclist killed crash]",https://www.eastbaytimes.com/2026/10/13/motorcyclist-killed-in-east-bay-free...,https://www.eastbaytimes.com/2026/10/13/motorcyclist-killed-in-east-bay-free...,https://mercurynews.com/2026/10/13/motorcyclist-killed-in-east-bay-freeway-c...,ok,None,page,True,EMERYVILLE — A motorcyclist was killed Tuesday morning in a collision with a...,"{'page_title': 'Motorcyclist killed in East Bay freeway collision', 'canonic...",None,california_confirmed,California,None,None,Interstate 80,2026-10-13T17:01:31+00:00,2026-10-13T07:

## 4. lead_verification
### `incident_analysis.json` — LLM evidence, quote-checked

In [13]:
ana = load("lead_verification", "incident_analysis")
print(ana["rules"], "\n")
show(ana["summary"])
ana_df = frame(ana["incidents"])
ana_df["n_people"] = ana_df["people"].str.len()
ana_df["n_sources"] = ana_df["sources"].str.len()
display(ana_df[["incident_id", "analysis_status", "in_scope", "evidence_status", "n_people", "n_sources", "scope_reasons", "open_questions", "error"]])

Every value comes from a quoted sentence found in the cited source. confirmed = official source or 2+ independent sites; single_source = one site; conflict = sources disagree; missing = not found (left null). Fault is reported as the sources state it, by statement type, and is never inferred from being struck, being a passenger, or another person's arrest or citation; it is not a legal determination. in_scope = no check failed (crash date in window, in California, a death reported). Residence is only what the sources state; home addresses are not searched for. 

{
  "incidents": 16,
  "by_analysis_status": {
    "no_text": 13,
    "analysed": 3
  },
  "by_evidence_status": {
    "single_source": 3
  },
  "in_scope": 3,
  "people_killed": 3
}


,incident_id,analysis_status,in_scope,evidence_status,n_people,n_sources,scope_reasons,open_questions,error
0,inc-8af338e5793d,no_text,None,insufficient,0,0,[],[no article text could be read],None
1,inc-aa2d0a31025b,no_text,None,insufficient,0,0,[],[no article text could be read],None
2,inc-1163e9e9a6ba,no_text,None,insufficient,0,0,[],[no article text could be read],None
3,inc-9173797236f6,no_text,None,insufficient,0,0,[],[no article text could be read],None
4,inc-a64bdbdbbc1d,no_text,None,insufficient,0,0,[],[no article text could be read],None
5,inc-ce9e077d6bc8,no_text,None,insufficient,0,0,[],[no article text could be read],None
6,inc-674acf242ee1,no_text,None,insufficient,0,0,[],[no article text could be read],None
7,inc-d18397b33e84,no_text,None,insufficient,0,0,[],[no article text could be read],None
8,inc-0addea92a067,no_text,None,insufficient,0,0,[],[no article text could be read],None
9,inc-2476bce1e57f,no_text,None,insufficient,0,0,[],[no article text could be read],None


In [14]:
# Every person across all analysed incidents
people = []
for rec in ana["incidents"]:
    for p in rec.get("people") or []:
        people.append({"incident_id": rec["incident_id"], **{k: v for k, v in p.items() if k != "summary"}, **{f"summary_{k}": v for k, v in (p.get("summary") or {}).items()}})
people_df = frame(people)
display(people_df)

,incident_id,person_ref,display_name,name_status,descriptor,field_status,evidence,fault,merge_note,summary_name,summary_age,summary_residence,summary_outcome,summary_role,summary_driver_of_vehicle,summary_fault_status
0,inc-64c24e170cb1,P1,unnamed male motorcyclist,unnamed,unnamed male motorcyclist,"{'full_name': 'missing', 'age': 'missing', 'residence': 'missing', 'outcome'...","{'full_name': {'value': None, 'status': 'missing', 'evidence': [], 'rejected...","{'status': 'under_investigation', 'statements': [{'statement': 'What caused ...",None,None,NaN,None,killed,motorcyclist,None,under_investigation
1,inc-64c24e170cb1,P2,Tesla driver,unnamed,Tesla driver,"{'full_name': 'missing', 'age': 'missing', 'residence': 'missing', 'outcome'...","{'full_name': {'value': None, 'status': 'missing', 'evidence': [], 'rejected...","{'status': 'under_investigation', 'statements': [{'statement': 'The Tesla dr...",None,None,NaN,None,uninjured,driver,None,under_investigation
2,inc-273f6dde6241,P1,unnamed male pedestrian,unnamed,unnamed male pedestrian,"{'full_name': 'missing', 'age': 'missing', 'residence': 'missing', 'outcome'...","{'full_name': {'value': None, 'status': 'missing', 'evidence': [], 'rejected...","{'status': 'undetermined', 'statements': [], 'rejected_statements': 1, 'note...",None,None,NaN,None,killed,pedestrian,None,undetermined
3,inc-043fce981ac8,P1,John Anderson,named,62-year-old male driver,"{'full_name': 'single_source', 'age': 'single_source', 'residence': 'missing...","{'full_name': {'value': 'John Anderson', 'status': 'single_source', 'evidenc...","{'status': 'under_investigation', 'statements': [{'statement': 'The crash wa...",None,John Anderson,62.0,None,killed,driver,None,under_investigation


In [15]:
# Full record of one analysed incident: change the index
rec = ana["incidents"][0]
show({k: v for k, v in rec.items() if k not in ("people", "sources", "articles")})
print("\nSources"); display(frame(rec["sources"]))
print("\nPeople")
for p in rec["people"]:
    show(p)

{
  "incident_id": "inc-8af338e5793d",
  "input_fingerprint": "8af338e5793d9d19",
  "analysis_status": "no_text",
  "analysed_at": "2026-10-14T01:21:50.925949+00:00",
  "error": null,
  "grouping": {
    "source_case_ids": [
      "case-4d4b12fbbe"
    ],
    "candidate_incident_dates": [
      {
        "date": "2026-10-10",
        "basis": "relative_to_publication",
        "text": "October 10",
        "evidence": "Riverside, CA – Motorcyclist Dylan Monterola-Alas Dies in Crash on Martin Luther King BoulevardThe crash happened at about 7:02 p.m. on October 10 on Martin Luther King Boulevard, about 53 feet east of Angelo Street, near Bordwell Park.",
        "article_id": "art-4d4b12fbbe4a"
      }
    ],
    "locations": [],
    "conflicts": [],
    "uncertainties": [
      "no article text could be read; evidence is RSS/listing metadata only"
    ],
    "grouping_rationale": [],
    "publishers": [
      "Facebook"
    ]
  },
  "analysis_log": {},
  "in_scope": null,
  "scope_reas

""



People


In [16]:
# analysis_log: follow-up searches, LLM web search, sources used
for rec in ana["incidents"]:
    print(rec["incident_id"]); show(rec["analysis_log"])

inc-8af338e5793d
{}
inc-aa2d0a31025b
{}
inc-1163e9e9a6ba
{}
inc-9173797236f6
{}
inc-a64bdbdbbc1d
{}
inc-ce9e077d6bc8
{}
inc-674acf242ee1
{}
inc-d18397b33e84
{}
inc-0addea92a067
{}
inc-2476bce1e57f
{}
inc-2750d1860a26
{}
inc-7d442f702ccb
{}
inc-0da8b8d90905
{}
inc-64c24e170cb1
{
  "already_fetched": 3,
  "followup_added": 0,
  "web_search_added": 0,
  "llm_calls": 1,
  "sources_used": [
    "https://www.nbcbayarea.com/news/local/emeryville-i-80-motorcyclist-killed-crash/4157340/",
    "https://www.eastbaytimes.com/2026/10/13/motorcyclist-killed-in-east-bay-freeway-collision-2/",
    "https://www.sfgate.com/news/bayarea/article/emeryville-fatal-motorcycle-crash-reported-on-22475029.php"
  ]
}
inc-273f6dde6241
{
  "already_fetched": 1,
  "followup_added": 0,
  "web_search_added": 0,
  "llm_calls": 1,
  "sources_used": [
    "https://www.fox5atlanta.com/news/pedestrian-killed-dekalb-crash-other-wrecks-snarl-traffic.amp"
  ]
}
inc-043fce981ac8
{
  "already_fetched": 1,
  "followup_added": 4

## Funnel: records through the pipeline

In [17]:
display(pd.Series({
    "raw feed items": raw["total_items"],
    "collected records (after filters)": len(fetched["collected_records"]),
    "cases to open": len(to_open["confirmed"]) + len(to_open["doubtful"]),
    "articles extracted": len(art["articles"]),
    "incidents grouped": len(inc["incidents"]),
    "incidents analysed": len(ana["incidents"]),
}, name="count").to_frame())

,count
raw feed items,433
collected records (after filters),136
cases to open,108
articles extracted,135
incidents grouped,76
incidents analysed,16
